# USB Whisper Prediction on Random Chapter 1 Chunks

This notebook loads the fine-tuned Whisper model from `output/usb_whisper_finetuned`,
samples random 10-second windows from chapter 1 raw USB data, and shows:
1. true transcript (aligned from chapter transcript segments)
2. model predicted transcript
3. audio playback for raw USB chunk and demodulated ASR chunk

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.signal import butter, resample_poly, sosfiltfilt
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, Markdown, display

PROJECT_ROOT = Path('<REPO_ROOT>')
MODEL_DIR = PROJECT_ROOT / 'output/usb_whisper_finetuned'
USB_CH1_PATH = PROJECT_ROOT / 'May29_Alice/Chap_1_img.bin'
REFERENCE_JSON = PROJECT_ROOT / 'output/whisper_transcripts/chapter_01_whisper.json'

SAMPLE_RATE = 200_000
ASR_SAMPLE_RATE = 16_000
CENTER_FREQ = 20_000.0
BANDWIDTH = 2_500.0
BASEBAND_LPF_CUTOFF = 2_500.0

CHUNK_SECONDS = 10.0
N_RANDOM_CHUNKS = 1
SEED = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

required_paths = [MODEL_DIR, USB_CH1_PATH, REFERENCE_JSON]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f'Missing required path: {path}')

print(f'Using device: {DEVICE}')
print(f'Model dir: {MODEL_DIR}')
print(f'USB file: {USB_CH1_PATH}')
print(f'Reference transcript: {REFERENCE_JSON}')

In [6]:
def bandpass_sos(x: np.ndarray, fs: int, lowcut: float, highcut: float, order: int = 4) -> np.ndarray:
    nyq = fs / 2
    low = max(lowcut / nyq, 1e-6)
    high = min(highcut / nyq, 0.999999)
    sos = butter(order, [low, high], btype='band', output='sos')
    return sosfiltfilt(sos, x)


def demodulate_usb_to_16k(
    usb_data: np.ndarray,
    sample_rate: int,
    center_freq: float,
    bandwidth: float,
    baseband_lpf_cutoff: float,
    asr_sample_rate: int,
) -> np.ndarray:
    lowcut = center_freq - bandwidth / 2
    highcut = center_freq + bandwidth / 2
    usb_band = bandpass_sos(usb_data, sample_rate, lowcut, highcut, order=4)

    t = np.arange(len(usb_band), dtype=np.float64) / float(sample_rate)
    mixed_down = usb_band * np.cos(2 * np.pi * center_freq * t)

    nyq = sample_rate / 2
    lpf_norm = min(baseband_lpf_cutoff / nyq, 0.999999)
    sos_lpf = butter(6, lpf_norm, btype='low', output='sos')
    demod_baseband = sosfiltfilt(sos_lpf, mixed_down)

    demod_16k = resample_poly(demod_baseband, asr_sample_rate, sample_rate).astype(np.float32)
    peak = float(np.max(np.abs(demod_16k)) + 1e-12)
    return demod_16k / peak


def normalize_for_audio(x: np.ndarray) -> np.ndarray:
    peak = float(np.max(np.abs(x)) + 1e-12)
    return (x / peak).astype(np.float32)


def approximate_reference_text(start_sec: float, end_sec: float, segments: list[dict]) -> str:
    pieces = []
    for seg in segments:
        seg_start = float(seg.get('start', 0.0))
        seg_end = float(seg.get('end', 0.0))
        overlap_start = max(start_sec, seg_start)
        overlap_end = min(end_sec, seg_end)

        if overlap_end <= overlap_start:
            continue

        text = seg.get('text', '').strip()
        if not text:
            continue

        words = text.split()
        seg_duration = max(seg_end - seg_start, 1e-6)
        rel_start = (overlap_start - seg_start) / seg_duration
        rel_end = (overlap_end - seg_start) / seg_duration
        w0 = int(len(words) * rel_start)
        w1 = max(w0 + 1, int(len(words) * rel_end))

        snippet = ' '.join(words[w0:w1]).strip()
        if snippet:
            pieces.append(snippet)

    return ' '.join(pieces).strip()


In [7]:
processor = WhisperProcessor.from_pretrained(str(MODEL_DIR))
model = WhisperForConditionalGeneration.from_pretrained(str(MODEL_DIR)).to(DEVICE)
model.eval()
model.config.forced_decoder_ids = None
if hasattr(model, 'generation_config'):
    model.generation_config.forced_decoder_ids = None

usb_data = np.fromfile(USB_CH1_PATH, dtype=np.float32)
usb_duration_sec = len(usb_data) / SAMPLE_RATE
print(f'Loaded USB samples: {len(usb_data):,} ({usb_duration_sec:.2f} sec)')

demod_16k = demodulate_usb_to_16k(
    usb_data=usb_data,
    sample_rate=SAMPLE_RATE,
    center_freq=CENTER_FREQ,
    bandwidth=BANDWIDTH,
    baseband_lpf_cutoff=BASEBAND_LPF_CUTOFF,
    asr_sample_rate=ASR_SAMPLE_RATE,
)
print(f'Demodulated length: {len(demod_16k):,} ({len(demod_16k) / ASR_SAMPLE_RATE:.2f} sec)')

with REFERENCE_JSON.open('r', encoding='utf-8') as f:
    reference_segments = json.load(f).get('segments', [])

print(f'Reference segments loaded: {len(reference_segments)}')

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 888.44it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


Loaded USB samples: 128,165,136 (640.83 sec)
Demodulated length: 10,253,211 (640.83 sec)
Reference segments loaded: 22


In [8]:
rng = np.random.default_rng(SEED)
max_start = max(0.0, usb_duration_sec - CHUNK_SECONDS)
start_times = sorted(rng.uniform(0.0, max_start, size=N_RANDOM_CHUNKS))

records = []
playback_rows = []

for chunk_id, start_sec in enumerate(start_times):
    end_sec = start_sec + CHUNK_SECONDS

    usb_start = int(start_sec * SAMPLE_RATE)
    usb_end = int(end_sec * SAMPLE_RATE)
    asr_start = int(start_sec * ASR_SAMPLE_RATE)
    asr_end = int(end_sec * ASR_SAMPLE_RATE)

    raw_chunk = usb_data[usb_start:usb_end].astype(np.float32)
    demod_chunk = demod_16k[asr_start:asr_end].astype(np.float32)
    demod_chunk = normalize_for_audio(demod_chunk)

    input_features = processor.feature_extractor(
        demod_chunk,
        sampling_rate=ASR_SAMPLE_RATE,
        return_tensors='pt',
    )['input_features'].to(DEVICE)

    with torch.no_grad():
        pred_ids = model.generate(input_features=input_features, max_new_tokens=128)

    pred_text = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()
    true_text = approximate_reference_text(start_sec, end_sec, reference_segments)

    records.append(
        {
            'chunk_id': chunk_id,
            'start_sec': round(start_sec, 3),
            'end_sec': round(end_sec, 3),
            'true_transcript': true_text,
            'predicted_transcript': pred_text,
        }
    )
    playback_rows.append(
        {
            'chunk_id': chunk_id,
            'start_sec': start_sec,
            'end_sec': end_sec,
            'true_transcript': true_text,
            'predicted_transcript': pred_text,
            'raw_chunk': normalize_for_audio(raw_chunk),
            'demod_chunk': demod_chunk,
        }
    )

results_df = pd.DataFrame(records)
display(results_df[['chunk_id', 'start_sec', 'end_sec', 'true_transcript', 'predicted_transcript']])

Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,chunk_id,start_sec,end_sec,true_transcript,predicted_transcript
0,0,488.231,498.231,"First, however, she waited for a few minutes t...",Alice's adventures in Wonderland by Lewis Carr...


In [ ]:
# --- CNN-CTC model inference on the demodulated chunks ---
import sys, json
import torch
import numpy as np
from pathlib import Path
sys.path.append(str(PROJECT_ROOT / "Training_Scripts"))
try:
    from cnn_asr_from_scratch import ConvCTCModel, compute_log_mel
except Exception as e:
    print("Could not import CNN utilities:", e)
    ConvCTCModel = None
    compute_log_mel = None

CNN_DIR = PROJECT_ROOT / 'output/cnn_asr'
tokenizer_path = CNN_DIR / 'tokenizer.json'
model_path = CNN_DIR / 'cnn_ctc_model.pth'

if not model_path.exists() or ConvCTCModel is None or compute_log_mel is None:
    print(f'CNN model or utilities not available at {CNN_DIR}; skipping CNN inference')
else:
    with tokenizer_path.open('r', encoding='utf-8') as f:
        tok = json.load(f)
    stoi = tok.get('stoi', {})
    itos_raw = tok.get('itos', {})
    # itos keys may be strings (from JSON); convert to int keys
    itos = {int(k): v for k, v in itos_raw.items()}
    num_classes = max(itos.keys()) + 1
    cnn = ConvCTCModel(n_mels=80, num_classes=num_classes, channels=64).to(DEVICE)
    cnn.load_state_dict(torch.load(model_path, map_location=DEVICE))
    cnn.eval()

    def ctc_greedy_decode(log_probs):
        # log_probs: (T, B, C) torch tensor
        with torch.no_grad():
            preds = log_probs.argmax(dim=2)  # (T, B)
            preds = preds.cpu().numpy()
        B = preds.shape[1]
        texts = []
        for b in range(B):
            seq = preds[:, b].tolist()
            out_chars = []
            prev = None
            for s in seq:
                if s == prev:
                    prev = s
                    continue
                if s != 0:
                    out_chars.append(itos.get(int(s), ''))
                prev = s
            texts.append(''.join(out_chars))
        return texts

    cnn_predictions = []
    for row in playback_rows:
        demod = row['demod_chunk']
        mel = compute_log_mel(demod, sr=ASR_SAMPLE_RATE, n_mels=80, hop_length=160)
        x = torch.from_numpy(mel).unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,n_mels,T)
        with torch.no_grad():
            logp = cnn(x)  # (T, B=1, C)
        texts = ctc_greedy_decode(logp)
        row['cnn_predicted_transcript'] = texts[0]
        cnn_predictions.append(texts[0])

    # Merge CNN predictions into results table
    import pandas as pd
    for i, rec in enumerate(records):
        rec['cnn_predicted_transcript'] = cnn_predictions[i] if i < len(cnn_predictions) else ''
    results_df = pd.DataFrame(records)
    display(results_df[['chunk_id','start_sec','end_sec','true_transcript','predicted_transcript','cnn_predicted_transcript']])

In [9]:
for row in playback_rows:
    display(
        Markdown(
            f"## Chunk {row['chunk_id']} ({row['start_sec']:.2f}s to {row['end_sec']:.2f}s)"
        )
    )
    display(Markdown(f"**True transcript:** {row['true_transcript'] or '[no aligned transcript found]'}"))
    display(Markdown(f"**Predicted transcript:** {row['predicted_transcript'] or '[model returned empty text]'}"))

    print('Raw USB chunk playback (200 kHz):')
    display(Audio(row['raw_chunk'], rate=SAMPLE_RATE))

    print('Demodulated chunk playback for ASR (16 kHz):')
    display(Audio(row['demod_chunk'], rate=ASR_SAMPLE_RATE))

## Chunk 0 (488.23s to 498.23s)

**True transcript:** First, however, she waited for a few minutes to see if she was going to shrink any further. She felt a little nervous about this. For my end, you know," said Alice to herself, in my going

**Predicted transcript:** Alice's adventures in Wonderland by Lewis Carroll, Chapter 2, The Pool of Tears.

Raw USB chunk playback (200 kHz):


Demodulated chunk playback for ASR (16 kHz):


If you want a larger or smaller sample set, update `N_RANDOM_CHUNKS`, `CHUNK_SECONDS`, and `SEED` in the config cell and rerun from the top.